In [68]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torch.utils.data import random_split, DataLoader
from torch.amp import autocast, GradScaler
import open3d as o3d
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.optim as optim
from sklearn.manifold import TSNE
import multiprocessing

DATA_PATH = "../data/buildings_pointcloud_ply"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Encoder

In [69]:
class PointNetPPEncoder(nn.Module):
    def __init__(self, latent_dim=256):
        super(PointNetPPEncoder, self).__init__()
        self.latent_dim = latent_dim

        # Shared MLPs for local feature extraction
        self.conv1 = nn.Conv1d(3, 96, 1)
        self.conv2 = nn.Conv1d(96, 192, 1)
        self.conv3 = nn.Conv1d(192, 384, 1)
        self.conv4 = nn.Conv1d(384, 768, 1)
        self.conv5 = nn.Conv1d(768, 1536, 1)
        self.conv6 = nn.Conv1d(1536, 3072, 1)

        # Fully connected layers for global features
        self.fc1 = nn.Linear(3072, 1536)
        self.fc2 = nn.Linear(1536, 768)
        self.fc3 = nn.Linear(768, 384)
        self.fc_mu = nn.Linear(384, latent_dim)       # Mean of latent distribution
        self.fc_log_sigma = nn.Linear(384, latent_dim)  # Log variance of latent distribution

    def forward(self, x):
        # Input x: (batch_size, num_points, 3)
        x = x.permute(0, 2, 1)  # Change to (batch_size, 3, num_points)

        # Apply shared MLPs
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))

        # Global max pooling
        x = torch.max(x, dim=2)[0]  # (batch_size, 256)

        # Fully connected layers for latent representation
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        mu = self.fc_mu(x)           # (batch_size, latent_dim)
        log_sigma = self.fc_log_sigma(x)  # (batch_size, latent_dim)
        return mu, log_sigma

Decoder

In [70]:
class PointNetPPDecoder(nn.Module):
    def __init__(self, latent_dim=128, num_points=4096):
        super(PointNetPPDecoder, self).__init__()
        self.latent_dim = latent_dim
        self.num_points = num_points

        # Fully connected layers to create an initial coarse point cloud
        self.fc1 = nn.Linear(latent_dim, 384)
        self.fc2 = nn.Linear(384, 768)
        self.fc3 = nn.Linear(768, 1536)  # Coarse point cloud (e.g., 256 points * 3)

        # Upsampling layers for finer details
        self.fc4 = nn.Linear(1536, 3072)
        self.fc5 = nn.Linear(3072, 6144)
        self.fc6 = nn.Linear(6144, num_points * 3) # Final high-resolution output

    def forward(self, z):
        # Input z: (batch_size, latent_dim)
        x = F.relu(self.fc1(z))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))  # Coarse point cloud

        # Upsample to finer resolution
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = self.fc6(x)  # (batch_size, num_points * 3)

        # Reshape to point cloud format
        x = x.view(-1, self.num_points, 3)  # (batch_size, num_points, 3)
        return x

Reparameterization trick

In [71]:
def reparameterize(mu, log_sigma):
    """
    Reparameterization trick to sample latent variable z.
    z = mu + eps * exp(log_sigma * 0.5)
    """
    std = torch.exp(0.5 * log_sigma)
    epsilon = torch.randn_like(std)
    return mu + epsilon * std

Encoder + Decoder + Reparameterization Trick = Autoencoder (in our case, Variational Autoencoder because we have mu and log sigma)

In [72]:
class PointNetPPAutoencoder(nn.Module):
    def __init__(self, latent_dim=256, num_points=4096):
        super(PointNetPPAutoencoder, self).__init__()
        self.encoder = PointNetPPEncoder(latent_dim)
        self.decoder = PointNetPPDecoder(latent_dim, num_points)

    def forward(self, x):
        """
        Returns:
          reconstructed: (batch_size, num_points, 3)
          mu: (batch_size, latent_dim)
          log_sigma: (batch_size, latent_dim)
        """
        mu, log_sigma = self.encoder(x)
        z = reparameterize(mu, log_sigma)  # <-- Use the reparam trick
        reconstructed = self.decoder(z)
        return reconstructed, mu, log_sigma

Loss Functions

In [73]:
def chamfer_distance(x, y):
    """
    Compute Chamfer Distance between two point clouds x and y.
    """
    distances = torch.cdist(x, y, p=2)
    min_dist_x = torch.min(distances, dim=2)[0]
    min_dist_y = torch.min(distances, dim=1)[0]
    return torch.mean(min_dist_x) + torch.mean(min_dist_y)

def kl_divergence(mu, log_sigma):
    """
    Compute the KL divergence between the encoder's latent distribution and a standard Gaussian.
    Args:
        mu: Mean of the latent distribution (batch_size, latent_dim)
        log_sigma: Log of standard deviation (batch_size, latent_dim)
    Returns:
        kl_loss: The KL divergence loss (scalar)
    """
    kl_loss = -0.5 * torch.sum(1 + log_sigma - mu.pow(2) - log_sigma.exp(), dim=1)
    return torch.mean(kl_loss)  # Return the batch mean

In [74]:
class PointCloudDataset(data.Dataset):
    def __init__(self, folder_path, num_points=4096):
        self.folder_path = folder_path
        self.num_points = num_points
        self.files = [f for f in os.listdir(folder_path) if f.endswith('.ply')]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = os.path.join(self.folder_path, self.files[idx])
        pcd = o3d.io.read_point_cloud(file_path)
        points = np.asarray(pcd.points)

        # Scale and center the point cloud to fit within a unit cube or sphere:
        points -= np.mean(points, axis=0)  # Center
        # points /= np.max(np.linalg.norm(points, axis=1))  # I chose not to scale since I would like to see if my latent space can predict for height

        # Randomly sample or pad points to ensure uniformity
        if points.shape[0] > self.num_points:
            indices = np.random.choice(points.shape[0], self.num_points, replace=False)
            points = points[indices]
        elif points.shape[0] < self.num_points:
            pad_size = self.num_points - points.shape[0]
            pad_points = np.zeros((pad_size, 3))
            points = np.vstack((points, pad_points))

        return torch.tensor(points, dtype=torch.float32)


Data Loader

In [75]:
def get_optimal_num_workers(fraction: float = 0.75, max_cap: int = 16):
    """
    Automatically determine optimal number of workers for DataLoader.
    Args:
        fraction (float): Fraction of total CPU cores to use.
        max_cap (int): Upper limit on number of workers to prevent overload.
    Returns:
        int: Recommended num_workers value.
    """
    total_cores = multiprocessing.cpu_count()
    recommended = int(total_cores * fraction)
    return min(recommended, max_cap)

In [76]:
def get_data_loaders(folder_path=DATA_PATH,
                     num_points=4096,
                     batch_size=32,
                     train_ratio=0.9,
                     random_seed=42):  # Added a seed parameter for reproducibility
    # Set the random seed for reproducibility
    torch.manual_seed(random_seed)

    dataset = PointCloudDataset(folder_path, num_points=num_points)
    train_size = int(train_ratio * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    optimal_workers = get_optimal_num_workers()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=optimal_workers,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=optimal_workers,
        pin_memory=True
    )

    print(f"Dataset split: {len(train_dataset)} train samples, {len(test_dataset)} test samples")
    return train_loader, test_loader



Define Training Loop

In [77]:
def train_model(folder_path=DATA_PATH,
                latent_dim=256,
                num_points=4096,
                batch_size=16,
                num_epochs=50,
                learning_rate=0.001,
                beta=1.0,
                device=DEVICE):

    # Prepare data
    train_loader, test_loader = get_data_loaders(
        folder_path=folder_path,
        num_points=num_points,
        batch_size=batch_size,
        train_ratio=0.8
    )

    # Initialize model
    model = PointNetPPAutoencoder(latent_dim=latent_dim, num_points=num_points)
    model.to(device)

    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    loss_history = []
    test_loss_history = []

    model_save_path = "weights/pointnetpp_B3-VAE.pth"
    best_loss = float('inf')

    print("Starting training...")
    scaler = GradScaler()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", leave=False, dynamic_ncols=True)

        for batch in progress_bar:
            batch = batch.to(device)
            optimizer.zero_grad()

            with autocast(DEVICE.type):
                reconstructed, mu, log_sigma = model(batch)
                recon_loss = chamfer_distance(reconstructed, batch)
                kld_loss = kl_divergence(mu, log_sigma)
                loss = recon_loss + beta * kld_loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            progress_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})

        avg_epoch_loss = epoch_loss / len(train_loader)
        loss_history.append(avg_epoch_loss)

        # Evaluate on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                with autocast(DEVICE.type):
                    reconstructed, mu, log_sigma = model(batch)
                    recon_loss = chamfer_distance(reconstructed, batch)
                    kld_loss = kl_divergence(mu, log_sigma)
                    loss = recon_loss + beta * kld_loss
                    test_loss += loss.item()

        avg_test_loss = test_loss / len(test_loader)
        test_loss_history.append(avg_test_loss)
        tqdm.write(f"Epoch {epoch + 1}/{num_epochs}, Test Loss: {avg_test_loss:.4f}")

        # Save the trained model if test loss improves
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            torch.save(model.state_dict(), model_save_path)
            tqdm.write(f"Model improved and saved to {model_save_path}")

    # Plot loss history
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, num_epochs + 1), loss_history, label='Train Loss', marker='o')
    plt.plot(range(1, num_epochs + 1), test_loss_history, label='Test Loss', marker='s')
    plt.title("Training and Test Loss Over Epochs (Chamfer + KL)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.show()

Train

In [ ]:
train_model(
    folder_path=DATA_PATH,
    latent_dim=10,
    num_points=4096,
    batch_size=16,
    num_epochs=50,
    learning_rate=0.001,
    beta=3.0,
    device=DEVICE
)

Dataset split: 3205 train samples, 802 test samples
Starting training...


Epoch 1/50, Test Loss: 5.3856
Model improved and saved to weights/pointnetpp_B3-VAE.pth


Epoch 2/50, Test Loss: 5.3421
Model improved and saved to weights/pointnetpp_B3-VAE.pth


Epoch 3/50, Test Loss: 5.4075


Epoch 4/50, Test Loss: 5.2830
Model improved and saved to weights/pointnetpp_B3-VAE.pth


...

Epoch 6/50, Test Loss: 5.2904


Epoch 7/50, Test Loss: 5.3243


Epoch 8/50, Test Loss: 5.2716
Model improved and saved to weights/pointnetpp_B3-VAE.pth


Epoch 9/50, Test Loss: 5.2593
Model improved and saved to weights/pointnetpp_B3-VAE.pth


KeyboardInterrupt: 

Load the model and play around

In [ ]:
def visualize_input_and_reconstructed(model, test_loader, device=DEVICE):
    model = model.to(device)
    model.eval()

    with torch.no_grad():
        # Get the first batch from the test_loader
        for batch in test_loader:
            # Shape of batch: (batch_size, num_points, 3)
            batch = batch.to(device)
            break  # Just get the first batch

        # Let's visualize the first point cloud in this batch
        input_pc = batch[0]  # shape: (num_points, 3)
        input_np = input_pc.cpu().numpy()  # Convert to numpy for open3d

        # Encode and decode the point cloud
        mu, log_sigma = model.encoder(input_pc.unsqueeze(0))  # add batch dim => (1, num_points, 3)
        z = reparameterize(mu, log_sigma)
        reconstructed = model.decoder(z)  # shape: (1, num_points, 3)

        # Convert reconstructed to numpy
        reconstructed_np = reconstructed[0].cpu().numpy()

    try:
        o3d.visualization.webrtc_server.enable_webrtc()
        using_webrtc = True
    except Exception as e:
        print("WebRTC not supported or already running:", e)
        using_webrtc = False

    # Create Open3D point cloud for the input
    pcd_input = o3d.geometry.PointCloud()
    pcd_input.points = o3d.utility.Vector3dVector(input_np)
    # Color the input cloud red
    pcd_input.paint_uniform_color([1.0, 0.0, 0.0])

    # Create Open3D point cloud for the reconstructed shape
    pcd_reconstructed = o3d.geometry.PointCloud()
    pcd_reconstructed.points = o3d.utility.Vector3dVector(reconstructed_np)
    # Color the reconstructed cloud green
    pcd_reconstructed.paint_uniform_color([0.0, 1.0, 0.0])

    # (Optional) Shift the reconstructed cloud so it’s easier to see side by side
    shift_along_x = 1.0
    pcd_reconstructed.translate((shift_along_x, 0, 0))

    # Visualize using modern Open3D backend
    if using_webrtc:
        o3d.visualization.draw([pcd_input, pcd_reconstructed])
    else:
        # Fallback if WebRTC not available
        o3d.visualization.draw_geometries([pcd_input, pcd_reconstructed])

In [ ]:
model = PointNetPPAutoencoder(latent_dim=128, num_points=4096)
model.load_state_dict(torch.load("pointnetpp_B3-VAE.pth", map_location=torch.device(DEVICE), weights_only=True))

model.to(DEVICE)

train_loader, test_loader = get_data_loaders(
    folder_path=DATA_PATH,
    num_points=4096,
    batch_size=4,  # or 1, if you want just one per batch
    train_ratio=0.9
)

visualize_input_and_reconstructed(model, test_loader, device=DEVICE)

In [ ]:
# ---------------------------
# 1. Load your trained (beta-)VAE model
# ---------------------------

# Load model
model = PointNetPPAutoencoder(latent_dim=128, num_points=4096)
model.load_state_dict(torch.load("pointnetpp_B3-VAE.pth", map_location=torch.device(DEVICE), weights_only=True))
model.eval()

# ---------------------------
# 2. Prepare (or load) the dataset
# ---------------------------
# Assuming train_loader yields only `inputs`, which are point clouds.
all_latents = []

for inputs in train_loader:  # No labels
    inputs = inputs.to(DEVICE)  # Use 'cuda' if you have a GPU and the model is on GPU
    with torch.no_grad():
        z_mean, z_logvar = model.encoder(inputs)  # Call the encoder directly

        # Sample from Gaussian distribution using reparameterization trick
        std = torch.exp(0.5 * z_logvar)
        eps = torch.randn_like(std)  # Gaussian noise
        z_vec = z_mean + eps * std  # Sampled latent vector

    all_latents.append(z_vec.cpu().numpy())

all_latents = np.concatenate(all_latents, axis=0)  # shape: (N, latent_dim)

# ---------------------------
# 3. Apply t-SNE (or UMAP) to reduce dimensionality
# ---------------------------
tsne = TSNE(n_components=2, random_state=42)
latents_2d = tsne.fit_transform(all_latents)  # shape: (N, 2)

# ---------------------------
# 4. Plot the 2D t-SNE results
# ---------------------------
plt.figure(figsize=(8, 6))
plt.scatter(latents_2d[:, 0], latents_2d[:, 1], s=5)  # No labels, so just plot the points
plt.title("Latent Space Visualization (t-SNE)")
plt.xlabel("t-SNE Dim 1")
plt.ylabel("t-SNE Dim 2")
plt.show()